# Phase 11.6 — Recommendation Diversity and Constraints

Selection layer only (scores from 11.5 are unchanged).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.recommendation.baseline_ranker import BaselineRecommendationRanker
from productiq.recommendation.candidate_generation import generate_recommendation_candidates
from productiq.recommendation.catalog_embeddings import InMemoryProductEmbeddingProvider
from productiq.recommendation.config import RecommendationConfig
from productiq.recommendation.content_similarity_engine import ContentSimilarityEngine
from productiq.recommendation.contracts import RecommendationRequest, RecommendationType
from productiq.recommendation.feature_extractor import extract_features_for_candidates
from productiq.recommendation.generators import (
    AttributeRecommendationCandidateGenerator,
    BM25RecommendationCandidateGenerator,
    VectorRecommendationCandidateGenerator,
)
from productiq.recommendation.product_context import RecommendationProductContext
from productiq.recommendation.recommendation_selection import select_recommendations_with_diagnostics
from productiq.recommendation.seed_embedding import InMemorySeedEmbeddingProvider
from productiq.recommendation.seed_product import InMemoryRecommendationCatalog
from productiq.recommendation.selection_config import RecommendationSelectionConfig
from productiq.recommendation.selection_context import RecommendationSelectionContext
from productiq.representation.filtering import FilteringRepresentation
from productiq.representation.lexical import build_structured_lexical_text
from productiq.representation.schema import ProductRepresentation
from productiq.retrieval import LexicalIndexDocument, build_inverted_lexical_index
from productiq.retrieval.bm25 import BM25Scorer
from productiq.retrieval.semantic import SemanticSearchHit, SemanticVector

In [ ]:
products = [
    ProductRepresentation(
        product_id="P1",
        brand="nike",
        brand_normalized="nike",
        category_gender="Women",
        product_type="t_shirt",
        color=["black"],
    ),
    ProductRepresentation(
        product_id="P2",
        brand="nike",
        brand_normalized="nike",
        category_gender="Women",
        product_type="t_shirt",
        color=["white"],
    ),
    ProductRepresentation(
        product_id="P3",
        brand="nike",
        brand_normalized="nike",
        category_gender="Women",
        product_type="t_shirt",
        color=["red"],
    ),
]
catalog = InMemoryRecommendationCatalog(products)
seed = catalog.resolve_seed_product("P1")
by_id = {p.product_id: p for p in products}

In [ ]:
def _vec(i: int) -> SemanticVector:
    values = [0.0] * 384
    values[i % 384] = 1.0
    return SemanticVector(values=tuple(values))


embeddings = {p.product_id: _vec(idx) for idx, p in enumerate(products)}
lexical_index = build_inverted_lexical_index(
    [
        LexicalIndexDocument(
            product_id=p.product_id,
            lexical_text=build_structured_lexical_text(p),
        )
        for p in products
    ]
)

class _FakeVectorIndex:
    def search(self, query_vector, *, top_k):
        return (
            SemanticSearchHit(product_id="P2", similarity=0.95),
            SemanticSearchHit(product_id="P3", similarity=0.90),
        )[:top_k]


request = RecommendationRequest(
    seed_product_id="P1",
    recommendation_type=RecommendationType.SIMILAR,
    top_k=5,
    config=RecommendationConfig(),
)
generation = generate_recommendation_candidates(
    request,
    catalog=catalog,
    generators=(
        VectorRecommendationCandidateGenerator(
            vector_index=_FakeVectorIndex(),
            seed_embeddings=InMemorySeedEmbeddingProvider(embeddings),
        ),
        AttributeRecommendationCandidateGenerator(catalog),
        BM25RecommendationCandidateGenerator(lexical_index),
    ),
    filtering_by_product_id={},
)
similarities = ContentSimilarityEngine(
    embedding_provider=InMemoryProductEmbeddingProvider(embeddings),
    bm25_scorer=BM25Scorer(lexical_index),
).compute_for_candidates(seed, generation.candidates, by_id)
context_by_id = {c.product_id: RecommendationProductContext(product_id=c.product_id) for c in generation.candidates}
features = extract_features_for_candidates(
    seed_product=seed,
    candidates=generation.candidates,
    similarities_by_product_id={row.product_id: row for row in similarities},
    context_by_product_id=context_by_id,
)
ranked = BaselineRecommendationRanker().rank(
    features,
    top_k=5,
    seed_product_id=request.seed_product_id,
    recommendation_config=request.config,
)
ranked

In [ ]:
def _filtering(product_id: str) -> FilteringRepresentation:
    product = by_id[product_id]
    return FilteringRepresentation(
        product_id=product_id,
        source="ajio",
        brand=product.brand,
        brand_normalized=product.brand_normalized,
        category_gender=product.category_gender,
        product_type=product.product_type,
        color_raw="black",
        color_is_coded=True,
        color=product.color or ["black"],
        pattern=None,
        material=None,
        fit=None,
        sleeve=None,
        neckline=None,
        product_features=None,
        style_attributes=None,
        discount_price_inr=1000,
        original_price_inr=1200,
        price_anomaly=False,
    )


selection_config = RecommendationSelectionConfig(max_per_brand=1)
selection_context = RecommendationSelectionContext(
    seed_product_id="P1",
    filtering_by_product_id={row.product_id: _filtering(row.product_id) for row in ranked},
    product_by_product_id=by_id,
)
selection = select_recommendations_with_diagnostics(
    ranked,
    top_k=5,
    config=selection_config,
    context=selection_context,
)
selection